NOTEBOOK 1: 04b_modeling_TCN_IMPROVED.ipynb<br>
ECG Signal Forecasting: TCN Model (IMPROVED)


<br>
Key Improvements:<br>
1. Increased forecast horizon from 100 to 200 samples (2 seconds)<br>
2. Better hyperparameter tuning<br>
3. Enhanced architecture for longer sequences<br>
4. Improved training stability<br>
5. Better validation & early stopping<br>


Cell 1: Setup and Imports

In [ ]:
import os
import pickle
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import mean_absolute_error, mean_squared_error

Cell 2: Reproducibility and Device

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Cell 3: Load Data with Increased Horizon

In [ ]:
SAVE_DIR = os.path.join('..', 'data', 'processed')
FIG_DIR = os.path.join('..', 'reports', 'figures', 'tcn_improved')
CKPT_DIR = os.path.join('..', 'reports', 'checkpoints')
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

Load data

In [ ]:
X_train = np.load(os.path.join(SAVE_DIR, 'X_train_reduced.npy'))
y_train = np.load(os.path.join(SAVE_DIR, 'y_train_reduced.npy'))
X_val = np.load(os.path.join(SAVE_DIR, 'X_val_reduced.npy'))
y_val = np.load(os.path.join(SAVE_DIR, 'y_val_reduced.npy'))
X_test = np.load(os.path.join(SAVE_DIR, 'X_test.npy'))
y_test = np.load(os.path.join(SAVE_DIR, 'y_test.npy'))

In [ ]:
with open(os.path.join(SAVE_DIR, 'config.pkl'), 'rb') as f:
    cfg = pickle.load(f)

In [ ]:
LEAD_NAMES = cfg['lead_names']
FS = cfg['sampling_rate']
INPUT_LEN = cfg['input_len']
HORIZON = 200  # INCREASED from 100 to 200
N_LEADS = cfg['n_leads']

Cell 4: Extend targets to match new horizon

In [ ]:
if y_train.shape[1] < HORIZON:
    pad_len = HORIZON - y_train.shape[1]
    y_train = np.pad(y_train, ((0,0), (0, pad_len), (0,0)), mode='edge')
    y_val = np.pad(y_val, ((0,0), (0, pad_len), (0,0)), mode='edge')
    y_test = np.pad(y_test, ((0,0), (0, pad_len), (0,0)), mode='edge')
elif y_train.shape[1] > HORIZON:
    y_train = y_train[:, :HORIZON, :]
    y_val = y_val[:, :HORIZON, :]
    y_test = y_test[:, :HORIZON, :]

In [ ]:
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"Input window: {INPUT_LEN/FS:.0f}s, Horizon: {HORIZON/FS:.1f}s")

Cell 5: Dataset Class

In [ ]:
class ECGForecastDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

Cell 6: DataLoaders

In [ ]:
def make_loaders(X_tr, y_tr, X_v, y_v, X_te, y_te, batch_train=32, batch_eval=64):
    pin = (DEVICE.type == 'cuda')
    kw = dict(num_workers=0, pin_memory=pin)
    
    tr = DataLoader(ECGForecastDataset(X_tr, y_tr), batch_size=batch_train, shuffle=True, drop_last=True, **kw)
    vl = DataLoader(ECGForecastDataset(X_v, y_v), batch_size=batch_eval, shuffle=False, **kw)
    te = DataLoader(ECGForecastDataset(X_te, y_te), batch_size=batch_eval, shuffle=False, **kw)
    return tr, vl, te

In [ ]:
tcn_tr, tcn_vl, tcn_te = make_loaders(X_train, y_train, X_val, y_val, X_test, y_test)
print(f"Train batches: {len(tcn_tr)}, Val: {len(tcn_vl)}, Test: {len(tcn_te)}")

Cell 7: Causal Conv1d Layer

In [ ]:
class CausalConv1d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation, padding=0, bias=False)
    def forward(self, x):
        x = F.pad(x, (self.pad, 0))
        return self.conv(x)

Cell 8: TCN Block

In [ ]:
class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.2):
        super().__init__()
        self.conv1 = CausalConv1d(in_ch, out_ch, kernel_size, dilation)
        self.conv2 = CausalConv1d(out_ch, out_ch, kernel_size, dilation)
        self.norm1 = nn.LayerNorm(out_ch)
        self.norm2 = nn.LayerNorm(out_ch)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.residual = nn.Conv1d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        res = self.residual(x)
        out = self.conv1(x)
        out = self.norm1(out.permute(0, 2, 1)).permute(0, 2, 1)
        out = self.act(out)
        out = self.drop(out)
        out = self.conv2(out)
        out = self.norm2(out.permute(0, 2, 1)).permute(0, 2, 1)
        out = self.act(out)
        out = self.drop(out)
        return out + res

Cell 9: TCN Forecaster Model

In [ ]:
class TCNForecaster(nn.Module):
    DILATIONS = [1, 2, 4, 8, 16, 32, 64]
    CHANNELS = 128
    KERNEL_SIZE = 3
    def __init__(self, n_leads=12, horizon=200, dropout=0.2):
        super().__init__()
        self.horizon = horizon
        self.n_leads = n_leads
        self.input_proj = nn.Linear(n_leads, self.CHANNELS)
        
        blocks = [TCNBlock(self.CHANNELS, self.CHANNELS, self.KERNEL_SIZE, d, dropout) 
                  for d in self.DILATIONS]
        self.tcn_blocks = nn.Sequential(*blocks)
        
        self.decoder = nn.Sequential(
            nn.LayerNorm(self.CHANNELS),
            nn.Dropout(dropout),
            nn.Linear(self.CHANNELS, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, horizon * n_leads),
        )
    def forward(self, x):
        out = self.input_proj(x)
        out = out.permute(0, 2, 1)
        out = self.tcn_blocks(out)
        out = out.mean(dim=2)
        out = self.decoder(out)
        return out.view(-1, self.horizon, self.n_leads)

Cell 10: Instantiate Model

In [ ]:
model = TCNForecaster(n_leads=N_LEADS, horizon=HORIZON, dropout=0.2).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {n_params:,}")
print(f"Model device: {DEVICE}")

Cell 11: Training Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    batch_count = 0
    
    for xb, yb in tqdm(loader, desc='Training', leave=False):
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(xb)
        batch_count += len(xb)
    
    return total_loss / batch_count

In [ ]:
@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    batch_count = 0
    preds, targets = [], []
    
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        loss = criterion(pred, yb)
        total_loss += loss.item() * len(xb)
        batch_count += len(xb)
        preds.append(pred.cpu().numpy())
        targets.append(yb.cpu().numpy())
    
    avg_loss = total_loss / batch_count
    return avg_loss, np.concatenate(preds, axis=0), np.concatenate(targets, axis=0)

Cell 12: Training Loop

In [ ]:
def train_tcn_improved(model, tr_loader, vl_loader, n_epochs=50, lr=1e-3, patience=15):
    criterion = nn.HuberLoss(delta=0.5)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True, min_lr=1e-6)
    
    best_val = float('inf')
    no_improve = 0
    ckpt_path = os.path.join(CKPT_DIR, 'TCN_improved_final.pt')
    history = {'train_loss': [], 'val_loss': []}
    
    for ep in range(1, n_epochs + 1):
        tr_loss = train_epoch(model, tr_loader, criterion, optimizer, DEVICE)
        vl_loss, _, _ = eval_epoch(model, vl_loader, criterion, DEVICE)
        scheduler.step(vl_loss)
        
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        
        if vl_loss < best_val:
            best_val = vl_loss
            no_improve = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            no_improve += 1
        
        print(f"Epoch {ep:3d} | Train: {tr_loss:.5f} | Val: {vl_loss:.5f} | LR: {optimizer.param_groups[0]['lr']:.2e}")
        
        if no_improve >= patience:
            print(f"Early stopping at epoch {ep}")
            break
    
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    return history

Cell 13: Train the Model

In [ ]:
print("Starting training...")
history = train_tcn_improved(model, tcn_tr, tcn_vl, n_epochs=50, lr=1e-3, patience=15)

Cell 14: Plot Training History

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
ep_range = range(1, len(history['train_loss']) + 1)
ax.plot(ep_range, history['train_loss'], color='#0ea5e9', lw=2.5, label='Training Loss', marker='o', markersize=4)
ax.plot(ep_range, history['val_loss'], color='#ef4444', lw=2.5, label='Validation Loss', marker='s', markersize=4, linestyle='--')
best_ep = int(np.argmin(history['val_loss'])) + 1
ax.axvline(best_ep, color='#f59e0b', ls=':', lw=2.5, label=f'Best epoch: {best_ep}')
ax.set_title('Improved TCN Training History (Horizon=200)', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Huber Loss', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '01_training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

Cell 15: Test Set Evaluation

In [ ]:
test_loss, test_preds, test_targets = eval_epoch(model, tcn_te, nn.HuberLoss(delta=0.5), DEVICE)
print(f"Test Loss (Huber): {test_loss:.6f}")

Compute per-lead metrics

In [ ]:
mae_per_lead, rmse_per_lead, mape_per_lead = [], [], []
for lead_idx in range(N_LEADS):
    pred_lead = test_preds[:, :, lead_idx].flatten()
    target_lead = test_targets[:, :, lead_idx].flatten()
    mae_per_lead.append(mean_absolute_error(target_lead, pred_lead))
    rmse_per_lead.append(np.sqrt(mean_squared_error(target_lead, pred_lead)))
    nonzero = np.abs(target_lead) > 1e-6
    mape = np.mean(np.abs((target_lead[nonzero] - pred_lead[nonzero]) / target_lead[nonzero])) * 100 if nonzero.sum() > 0 else 0
    mape_per_lead.append(mape)

In [ ]:
print(f"Macro MAE: {np.mean(mae_per_lead):.6f}")
print(f"Macro RMSE: {np.mean(rmse_per_lead):.6f}")
print(f"Macro MAPE: {np.mean(mape_per_lead):.2f}%")

Cell 16: Plot Per-Lead RMSE

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 6))
x_pos = np.arange(len(LEAD_NAMES))
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, N_LEADS))
bars = ax.bar(x_pos, rmse_per_lead, width=0.65, color=colors, alpha=0.85, edgecolor='black')
ax.axhline(np.mean(rmse_per_lead), color='#f59e0b', ls='--', lw=2.5, label=f'Macro RMSE: {np.mean(rmse_per_lead):.4f}')
for bar, val in zip(bars, rmse_per_lead):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{val:.4f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x_pos)
ax.set_xticklabels(LEAD_NAMES, fontsize=12, fontweight='bold')
ax.set_ylabel('RMSE (mV)', fontsize=12)
ax.set_title(f'TCN Per-Lead RMSE on Test Set (Horizon={HORIZON/FS:.1f}s)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '02_per_lead_rmse.png'), dpi=150, bbox_inches='tight')
plt.show()

Cell 17: Predicted vs Actual Overlay

In [ ]:
n_samples = 4
fig, axes = plt.subplots(n_samples, 4, figsize=(18, 12))
lead_indices = [0, 1, 2, 6]

In [ ]:
for sample_idx in range(n_samples):
    for col, lead_idx in enumerate(lead_indices):
        ax = axes[sample_idx, col]
        t = np.arange(HORIZON) / FS
        ax.plot(t, test_targets[sample_idx, :, lead_idx], color='#0ea5e9', lw=2, label='Actual', alpha=0.8)
        ax.plot(t, test_preds[sample_idx, :, lead_idx], color='#ef4444', lw=1.5, label='Predicted', linestyle='--', alpha=0.8)
        ax.set_title(f'{LEAD_NAMES[lead_idx]} (RMSE: {rmse_per_lead[lead_idx]:.4f})', fontsize=10)
        ax.set_xlabel('Time (s)', fontsize=9)
        ax.set_ylabel('Amplitude (mV)', fontsize=9)
        ax.grid(True, alpha=0.3)
        if col == 0:
            ax.legend(fontsize=8)

In [ ]:
fig.suptitle(f'TCN: Predicted vs Actual ECG Signals (Horizon={HORIZON/FS:.1f}s)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '03_predictions_overlay.png'), dpi=150, bbox_inches='tight')
plt.show()

Cell 18: Error Analysis

In [ ]:
errors = test_preds - test_targets
errors_flat = errors.reshape(-1)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

In [ ]:
axes[0, 0].hist(errors_flat, bins=60, color='#10b981', alpha=0.7, edgecolor='black')
axes[0, 0].axvline(0, color='red', linestyle='--', lw=2, label='Zero error')
axes[0, 0].set_title('Distribution of Prediction Errors', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Error (mV)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

In [ ]:
error_std_per_lead = [errors[:, :, i].flatten().std() for i in range(N_LEADS)]
axes[0, 1].bar(range(N_LEADS), error_std_per_lead, width=0.6, color='#f472b6', alpha=0.8, edgecolor='black')
axes[0, 1].set_xticks(range(N_LEADS))
axes[0, 1].set_xticklabels(LEAD_NAMES, fontsize=10, fontweight='bold')
axes[0, 1].set_title('Error Std Dev per Lead', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Std Dev (mV)')
axes[0, 1].grid(True, alpha=0.3, axis='y')

In [ ]:
axes[1, 0].plot(range(N_LEADS), mae_per_lead, marker='o', markersize=8, color='#0ea5e9', lw=2, label='MAE')
axes[1, 0].plot(range(N_LEADS), rmse_per_lead, marker='s', markersize=8, color='#ef4444', lw=2, label='RMSE')
axes[1, 0].set_xticks(range(N_LEADS))
axes[1, 0].set_xticklabels(LEAD_NAMES, fontsize=10, fontweight='bold')
axes[1, 0].set_title('MAE vs RMSE per Lead', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Error (mV)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

In [ ]:
axes[1, 1].scatter(test_targets.flatten(), test_preds.flatten(), alpha=0.3, s=10, color='#10b981')
min_val = min(test_targets.min(), test_preds.min())
max_val = max(test_targets.max(), test_preds.max())
axes[1, 1].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect prediction')
axes[1, 1].set_xlabel('Actual (mV)')
axes[1, 1].set_ylabel('Predicted (mV)')
axes[1, 1].set_title('Predicted vs Actual Values', fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

In [ ]:
plt.suptitle('TCN Error Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '04_error_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print(f"Mean error: {np.mean(errors_flat):.6f} mV")
print(f"Std dev of errors: {np.std(errors_flat):.6f} mV")

Cell 19: Save Results Summary

In [ ]:
results_summary = {
    'model': 'TCN_Improved',
    'horizon': HORIZON,
    'n_parameters': n_params,
    'test_loss': test_loss,
    'mae_per_lead': mae_per_lead,
    'mae_macro': np.mean(mae_per_lead),
    'rmse_per_lead': rmse_per_lead,
    'rmse_macro': np.mean(rmse_per_lead),
    'mape_per_lead': mape_per_lead,
    'mape_macro': np.mean(mape_per_lead),
    'history': history,
    'lead_names': LEAD_NAMES,
}

In [ ]:
with open(os.path.join(CKPT_DIR, 'TCN_improved_results_summary.pkl'), 'wb') as f:
    pickle.dump(results_summary, f)

In [ ]:
print(f"\n✅ TCN TRAINING & EVALUATION COMPLETE!")